# Lab 39: The router's query-data lifecycle

The prototype trainset underrepresents messy real-user phrasing. Capture messy queries, dedup, triage by confidence to a review queue, retrain, and MEASURE whether it helped. Fill in the `TODO` cells; reference in `solution/`.

Closes the loop on [Lab 36](../36-training-the-router/).

## Step 0: Setup

In [ ]:
import json
import os
import pathlib
import re
import numpy as np
from dotenv import load_dotenv
here = pathlib.Path.cwd()
for parent in [here, *here.parents]:
    if (parent / ".env.example").exists():
        load_dotenv(parent / ".env")
        break
from sentence_transformers import SentenceTransformer
embedder=SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2",device="cpu")
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
print("ready")

## Step 1: Prototypes + captured messy queries

In [ ]:
# The prototype trainset (Lab 36) and a sample of CAPTURED queries - messy phrasings
# as they arrive in logs (lowercase, typos, fragments, "$$", "??"). In production these
# arrive UNLABELED; the `gold` field here stands in for what a human reviewer assigns.
with open("../36-training-the-router/router_trainset.jsonl") as f:
    proto = [json.loads(line) for line in f]
with open("./captured_queries.jsonl") as f:
    captured = [json.loads(line) for line in f]
with open("../34-rag-pattern-head-to-head/eval_set.jsonl") as f:
    clean_eval = [json.loads(line) for line in f]
CAT={"parametric":"parametric","global-theme":"global","multi-hop":"multihop",
     "off-corpus":"off_corpus_risk","specific-lookup":"specific","paraphrase":"specific"}

def embed(texts): return embedder.encode(texts,normalize_embeddings=True,convert_to_numpy=True,show_progress_bar=False)
Xp=embed([r["query"] for r in proto])
clf0=LogisticRegression(max_iter=2000,C=10,class_weight="balanced").fit(Xp,[r["route"] for r in proto])
print(f"{len(proto)} prototype queries, {len(captured)} captured (messy) queries")
print("trained the prototypes-only router (clf0)")

## Step 2: The distribution shift

Confidence collapses on messy queries — the evidence that the prototypes never saw their shape.

In [ ]:
# TODO: compute max predict_proba confidence for clf0 on the clean eval queries and on
# the captured messy queries; show messy mean < clean mean (the distribution shift). Then
# report clf0 accuracy on the messy gold labels (the number a new round must beat).
raise NotImplementedError

## Step 3: Dedup

Exact (normalized string) then near-duplicate (embedding cosine).

In [ ]:
# Step 1 of the lifecycle: DEDUP. Logs repeat. Two cheap layers:
#  (a) exact-ish: normalized string; (b) near-duplicate: high embedding cosine.
def norm(q):
    return re.sub(r"[^a-z0-9 ]", "", q.lower()).strip()
seen = set()
unique = []
for r in captured:
    k=norm(r["query"])
    if k not in seen:
        seen.add(k)
        unique.append(r)
# near-dup pass: drop a query whose max cosine to an already-kept query exceeds 0.95
Eu = embed([r["query"] for r in unique])
keep = []
kept_idx = []
for i,r in enumerate(unique):
    if kept_idx and float((Eu[kept_idx] @ Eu[i]).max()) > 0.95:
        continue
    keep.append(r)
    kept_idx.append(i)
print(f"captured {len(captured)} -> {len(unique)} after exact dedup -> {len(keep)} after near-dup")

## Step 4: Confidence triage (active learning)

Review the low-confidence tail — the highest-value labels.

In [ ]:
# Step 2: TRIAGE by confidence (active learning). Auto-accept high-confidence router
# labels; send the low-confidence tail to human review - that is where the model learns
# the most per label. Here the "review" returns the gold label.
THRESH=0.50
conf=clf0.predict_proba(embed([r["query"] for r in keep])).max(axis=1)
pred=clf0.predict(embed([r["query"] for r in keep]))
review_queue=[(r,p) for r,p,c in zip(keep, pred, conf, strict=False) if c<THRESH]
auto_accept=[(r,p) for r,p,c in zip(keep, pred, conf, strict=False) if c>=THRESH]
print(f"{len(review_queue)} -> human review (low confidence), {len(auto_accept)} auto-accepted")
# Newly labeled examples = reviewed (gold) + auto-accepted (router label). In practice
# you would spot-check the auto-accepted ones too.
newly_labeled=[{"query":r["query"],"route":r["gold"]} for r,_ in review_queue] + \
              [{"query":r["query"],"route":p} for r,p in auto_accept]
print(f"{len(newly_labeled)} newly labeled queries ready to merge")

## Step 5: Merge + retrain + measure

Augment with reviewed queries; hold out a messy test set to measure the effect.

In [ ]:
# TODO: stratified-split captured into aug/test halves. accA = clf0 accuracy on test.
# Train clfB on proto + reviewed(aug) and compute accB on the same test. Print A, B, delta.
raise NotImplementedError

## Step 6: Read it honestly

Promote the new model only on a measured lift.

In [ ]:
# Read the result HONESTLY. Adding data is not automatically an improvement:
#  - if delta > 0, the captured queries taught the boundary the prototypes missed - ship clfB.
#  - if delta <= 0, augmentation added noise or the test set is too small to resolve the
#    effect; do NOT ship on faith. Collect more, check per-route, or revisit labels.
# The discipline is the lesson: a "second training round" is an experiment you MEASURE on a
# held-out slice of the new distribution, not a guaranteed win.
accA = globals().get("accA", 0.0)
accB = globals().get("accB", 0.0)
if accB > accA:
    print("delta positive: the augmented router generalizes better to messy queries.")
else:
    print("delta not positive: more data did not help on this test slice - investigate before shipping.")
print("Either way: gate the promotion of clfB on a measured improvement, then re-derive")
print("the eval-gate baseline (Lab 38) because the model - and its routing accuracy - changed.")

## Step 7: The loop

In [ ]:
# The lifecycle, end to end:
#   capture logs -> dedup (string + near-dup) -> triage by confidence (review the low tail)
#   -> merge labels -> retrain -> MEASURE on held-out messy -> promote only on a real lift
#   -> re-baseline the eval gate (Lab 38).
# The confidence drop (clean vs messy) is the trigger that tells you a new round is due.
print("Watch the confidence distribution on live queries; when it sags, a new round is due.")

## What you built

The data lifecycle for the router: capture messy queries, dedup, triage by confidence to a review queue (active learning), merge labels, retrain, and measure the result on a held-out messy slice before promoting. The trigger for a new round is the confidence drop on live queries; the gate on shipping the new model is a measured improvement, after which you re-derive the [Lab 38](../38-calibrating-the-eval-gate/) baseline because the model changed.

**Where this simplifies:** `captured_queries.jsonl` is 36 hand-authored messy queries with gold labels — real logs are larger, unlabeled, and noisier, and a real review step costs annotator time (which is exactly why confidence triage matters); the augmentation effect is measured on a small held-out slice, so treat its sign as indicative, not definitive — with semantic embeddings the in-distribution augmentation effect is usually positive, but the discipline is to verify it, not assume it.

This closes Path 02's RAG arc: build the patterns (31-33), compare (34), route (35), train and harden the router (36), gate it in CI (37), calibrate the gate (38), and keep the router's data fresh (39).